# `theoryminer`, line by line

This notebook takes the package in `theoryminer/` apart, one function at a time.
It uses nothing from the other notebooks.

**How each function is explained**

1. **Source.** The notebook prints the function's code as it is in the file right now, with the file's line numbers.
2. **By hand.** It runs the function's lines again, one step at a time, on a small input, and prints every value in between.
3. **Check.** It calls the real function on the same input and checks with `assert` that it returns the same result.
   When an `assert` passes, it prints nothing. If a cell stops with an `AssertionError`, the package changed
   after this notebook was written.

**What the package does**

```
 PDF ──harvest()──► sentences ──causenet()──► relations (cause span → effect span)
                                                   │
                        ┌──────────────────────────┼─────────────────────────────┐
                        ▼                          ▼                             ▼
          standardize_constructs()      pairwise_grouping() / clusterer()        (spans)
          span → top-k taxonomy concepts   spans → groups of similar spans
                                                   │
                                   ┌───────────────┴───────────────┐
                                   ▼                               ▼
                         standardize_groups()               label_groups()
                         group → taxonomy concept           group → a label made from its own words
```

**Three data shapes pass between the functions.** Know these three and you can follow the package.

| Shape | Made by | Looks like |
|---|---|---|
| **row dict** | `harvest` (inside) | `{"doc_id", "page", "section", "text", "kept", "drop_reason", "flag"}` |
| **relation dict** | `causenet` | `{"rel_id", "sent_id", "doc_id", "page", "cause", "effect", "sentence"}` |
| **group dict** | `pairwise_grouping`, `clusterer` | `{span: {"group": id, "central": span}}` |

**Contents**

- Part 0: setup
- Part 1: `harvest.py`, from a PDF to clean sentences
- Part 2: `causenet.py`, from sentences to cause → effect pairs
- Part 3: `harmonizer/`, from raw spans to named constructs
  (`embeddings`, `spans`, `taxonomy`, `standardize`, `pairwise`, `cluster`, `labels`)
- Part 4: the whole chain on one page, and things the walkthrough turned up

**Time.** The first full run loads two models and embeds the ELSST thesaurus once (about 2 minutes on a CPU).
The thesaurus vectors are then saved in `theoryminer/taxonomy_cache/`, so later runs are faster.

> **Note (2026-09-26).** Parts of this notebook were written for an earlier data set: the
> *Oxford Handbook of Self-Determination Theory* and the file `Step_1.csv`. They are no longer part of
> the repository. The real examples now come from the open-access papers in `sample_files/`.
> Some numbers in the text belong to the old data. They change when you run the notebook again.

## Part 0 · Setup

The first code cell changes to the repository root, so the paths work from `tutorials/`. The `show()` helper prints a function's source
code from the package file, so what you read here is always the current code.

In [ ]:
import collections
import copy
import importlib
import inspect
import json
import math
import os
import re
import sys
import tempfile
import time

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_VERBOSITY"] = "error"

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)     # sklearn's HDBSCAN 'copy' notice
warnings.filterwarnings("ignore", message=".*IProgress.*")      # tqdm's notebook-widget notice

import numpy as np
import pandas as pd

# Jupyter starts a notebook in its own folder. The paths below start at the repository root.
if os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))          # the package folder is in the repository root
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)


def show(obj):
    # Print the source of a package function, with its line numbers in the file.
    lines, start = inspect.getsourcelines(obj)
    path = inspect.getsourcefile(obj)
    print("# " + os.path.relpath(path))
    for number, line in enumerate(lines, start):
        print(f"{number:4d} | {line}", end="")

**A trap in the imports.** `theoryminer/__init__.py` contains `from .harvest import harvest`. That line
puts the *function* `harvest` in the package attribute that had the *module* `harvest`. So
`from theoryminer import harvest` gives you the function, and `import theoryminer.harvest as H`
also gives you the **function**. To get the module (the file, with its helpers), use
`importlib.import_module`, which reads the module from `sys.modules`:

In [ ]:
from theoryminer import (harvest, causenet, standardize_constructs, standardize_groups,
                               pairwise_grouping, clusterer, label_groups)

H  = importlib.import_module("theoryminer.harvest")                # the file harvest.py
C  = importlib.import_module("theoryminer.causenet")               # the file causenet.py
E  = importlib.import_module("theoryminer.harmonizer.embeddings")
S  = importlib.import_module("theoryminer.harmonizer.spans")
T  = importlib.import_module("theoryminer.harmonizer.taxonomy")
St = importlib.import_module("theoryminer.harmonizer.standardize")
P  = importlib.import_module("theoryminer.harmonizer.pairwise")
Cl = importlib.import_module("theoryminer.harmonizer.cluster")
L  = importlib.import_module("theoryminer.harmonizer.labels")

print(type(harvest), "<- what `from theoryminer import harvest` gives")
print(type(H), "<- what importlib gives")

---
# Part 1 · `harvest.py`: from a PDF to clean sentences

`harvest(source)` reads one PDF, or every PDF in a folder. For each PDF, `_harvest_one` calls four small helpers in order:

1. `_read_pages`: get the text of each page as markdown (from the PDF, or from a cache file).
2. `_select_pages`: select the pages you asked for.
3. `_page_rows`: make **one row per line** of text.
4. `_run_steps`: run the **steps**, an ordered list of small functions, over the rows.

Then `harvest` separates the kept sentences from the dropped rows, prints a summary, and returns
`{"sentences": [...], "dropped": [...]}`.

Helpers 1 to 3 only move data around. The work is in helper 4. It runs ten steps in this order:

| # | step | works on | what it does |
|---|---|---|---|
| 1 | `fix_unicode` | lines | changes curly quotes, dashes, odd spaces and ligatures to plain ASCII |
| 2 | `strip_markdown` | lines | removes `[link](url)` and `*emphasis*` markers and keeps the words |
| 3 | `drop_running_heads` | lines | drops lines that repeat on 3 or more pages |
| 4 | `drop_junk_lines` | lines | drops headings, tables, page numbers, DOIs and reference sections |
| 5 | `join_paragraphs` | lines → paragraphs | joins lines into paragraphs and repairs page breaks and hyphens |
| 6 | `split_sentences` | paragraphs → sentences | uses the Punkt splitter, which knows abbreviations like *e.g.* |
| 7 | `strip_citations` | sentences | removes `(Ryan & Deci, 2017)` |
| 8 | `strip_stats` | sentences | removes `(p < .05, n = 120)` |
| 9 | `drop_junk_sentences` | sentences | drops captions, index entries and mangled text; flags fragments |
| 10 | `filter_length` | sentences | drops sentences shorter than 4 or longer than 80 words |

We first look at the one data shape all of these steps share. Then we look at the constants and helpers at the
top of the file. Then we run the ten steps by hand on a small made-up book, and last on a real paper from `sample_files/`.

## 1.1 The row dict: the only data shape inside `harvest`

Every step **takes a list of row dicts and returns a list of row dicts**. That is the whole contract, and it is
why you can reorder the steps, remove one, or add your own.

A step **never deletes a row**. It sets `kept = False` and writes the reason in `drop_reason`. The row stays in
the list, so at the end every drop can be shown with its reason. The project needs this rule: a sentence lost
while cleaning would otherwise look like a "pipeline miss" in the evaluation.

`flag` is for doubtful rows that are **kept** (`"fragment"`, `"long"`).

In [ ]:
row = {"doc_id": "toy", "page": 1, "section": "", "text": "Autonomy supports motivation.",
       "kept": True, "drop_reason": "", "flag": ""}
row

## 1.2 The constants at the top of the file

### The Unicode table `TRANS`

PDFs are full of characters that look like ASCII but are not: curly quotes, en and em dashes, non-breaking
spaces, soft hyphens and zero-width spaces. Four dicts map each of them to a plain character, or to `""` to
delete it. `str.maketrans` merges the four dicts into one translation table.

In [ ]:
print("QUOTES:", H.QUOTES)
print("DASHES:", H.DASHES)
print("SPACES:", {repr(k): v for k, v in H.SPACES.items()})
print("KILL  :", {repr(k): v for k, v in H.KILL.items()})

`fix_unicode` does two things in a row: `unicodedata.normalize("NFKC", text)` splits ligatures such as `ﬁ` into `f` + `i`
(NFKC = "compatibility" normalisation), then `.translate(TRANS)` applies the table. The code below does the two
operations one after the other on a string that contains the usual problems:

In [ ]:
import unicodedata

messy = "The eﬀect of teachers’ “autonomy support” is signiﬁcant – and last­ing​."
after_nfkc = unicodedata.normalize("NFKC", messy)
after_trans = after_nfkc.translate(H.TRANS)

print("input      :", repr(messy))
print("after NFKC :", repr(after_nfkc))
print("after TRANS:", repr(after_trans))
print()
print("characters that changed:")
for ch in dict.fromkeys(messy):
    if not ch.isascii():
        print(f"   U+{ord(ch):04X} {unicodedata.name(ch, '?'):<32} -> {unicodedata.normalize('NFKC', ch).translate(H.TRANS)!r}")

### The line-level patterns

`drop_junk_lines` uses these regexes to recognise lines that are layout, not prose. `.match` checks from the
**start** of the line, and patterns ending in `$` must also reach the end. A ✓ means that the pattern matches the line:

In [ ]:
line_patterns = {"HEADING": H.RE_HEADING, "TABLE": H.RE_TABLE, "DOI": H.RE_DOI,
                 "NOSPACE": H.RE_NOSPACE, "PAGENUM": H.RE_PAGENUM, "SECT_REF": H.RE_SECT_REF,
                 "CHAPTER": H.RE_CHAPTER, "INDEXY": H.RE_INDEXY}
test_lines = ["## Autonomy and Motivation", "## References", "| Study | N | Effect |",
              "https://doi.org/10.1037/0003-066X.55.1.68", "10.1037/0003-066X.55.1.68",
              "pp.123-145", "xii", "12", "civil", "CHAPTER", "Contents",
              "Autonomy supports motivation."]
pd.DataFrame([{"line": line, **{name: "✓" if p.match(line) else "" for name, p in line_patterns.items()}}
              for line in test_lines])

What to see in the table:

- `## References` matches **two** patterns. The order of the `if/elif` chain in `drop_junk_lines` decides what
  happens: the reference check comes first, so the line starts a reference section.
- `RE_NOSPACE` means "one token, no spaces", so it matches **every** one-word line (`xii`, `civil`, `CHAPTER`, the
  URL). `drop_junk_lines` only uses it after the other checks, and only for a line that **also contains a digit**
  (`ref_fragment`). So `pp.123-145` is dropped, and an ordinary one-word line is left for the later rules.
- `RE_PAGENUM` is `[ivxlcdm\d]{1,6}`, which covers roman numerals, so a line that reads only `civil` also matches.
  This does no harm in practice, because a line of real prose is longer than one word.
- Plain prose matches nothing, which is the point.

### The sentence-level patterns

These patterns look **inside** a sentence (`.search` / `.findall`), not only at its start:

In [ ]:
examples = {
    "RE_YEAR":      (H.RE_YEAR,      "as shown in 2017a and in the 1990s, but not in 2150"),
    "RE_PAREN":     (H.RE_PAREN,     "persistence (Ryan & Deci, 2017) and effort (p < .05)"),
    "RE_BRACK":     (H.RE_BRACK,     "see the model [Figure 2] and [12]"),
    "RE_STAT":      (H.RE_STAT,      "p < .05, n = 120, SD = 1.2, 95% CI"),
    "RE_PAGERANGE": (H.RE_PAGERANGE, "autonomy, 12-14, 55-60, 101-110"),
}
for name, (pattern, text) in examples.items():
    print(f"{name:<13} on {text!r}")
    print(f"{'':13}    findall -> {pattern.findall(text)}")

print()
for text in ["Figure 3. The continuum.", "Keywords: autonomy, need", "Table 2 shows", "The figure 3 shows"]:
    print(f"RE_LABEL.match({text!r:<30}) -> {bool(H.RE_LABEL.match(text))}")
for text in ["It ends here.", 'It ends here."', "It ends here.)", "It does not end"]:
    print(f"RE_TERMINAL.search({text!r:<18}) -> {bool(H.RE_TERMINAL.search(text))}")

`RE_LABEL` only fires at the **start** of a sentence, so "The figure 3 shows" is prose and stays.
`RE_TERMINAL` accepts a closing quote or bracket after the final `.`/`!`/`?`.

### The sentence splitter `SPLITTER`

NLTK's Punkt splitter guesses where a sentence ends. Without help, it treats the full stop in *e.g.*, *et al.*
or *Fig.* as the end of a sentence. `ABBREV` is a list of abbreviations that do **not** end a sentence. The code
puts the list into `PunktParameters` and builds the splitter once, when the module is imported.

In [ ]:
from nltk.tokenize.punkt import PunktSentenceTokenizer

text = ("Rewards can undermine interest, cf. the early studies. Deci et al. found this in "
        "play, e.g. with puzzles. See Fig. 2 for the design.")
print("Punkt with no abbreviation list:")
for s in PunktSentenceTokenizer().tokenize(text):
    print("   |", s)
print("SPLITTER (Punkt + ABBREV):")
for s in H.SPLITTER.tokenize(text):
    print("   |", s)
print()
print(len(H.ABBREV), "abbreviations, for example:", H.ABBREV[:12])

### The thresholds

The last constants are the thresholds of the junk rules. We meet each one where it is used.

In [ ]:
print("MIN_ALPHA  =", H.MIN_ALPHA, " a sentence needs at least this share of letters and spaces")
print("MAX_DIGIT  =", H.MAX_DIGIT, " a sentence with a larger share of digits is junk")
print("MAX_SPACED =", H.MAX_SPACED, "share of broken 1-2 letter tokens that marks a mangled line")
print("LONG_FLAG  =", H.LONG_FLAG, "   a sentence with more words gets the flag 'long'")
print("LEGIT_SHORT (real short words):", sorted(H.LEGIT_SHORT)[:20], "...")

## 1.3 The small helpers

### `_alpha_ratio`: what share of the text is letters or spaces

In [ ]:
show(H._alpha_ratio)

In [ ]:
for text in ["Autonomy supports motivation.", "pp. 12-14, 2019; 33-35", "3.14 / 2.71 = 1.16", ""]:
    letters = sum(ch.isalpha() for ch in text)
    spaces = sum(ch.isspace() for ch in text)
    by_hand = (letters + spaces) / len(text) if text else 0.0
    assert by_hand == H._alpha_ratio(text)
    print(f"{text!r:<32} letters={letters:>2} spaces={spaces:>2} length={len(text):>2}  -> {by_hand:.2f}")

### `_spaced_letter_ratio`: detect text that the PDF broke into pieces

Some PDFs set text with letter spacing. The extractor then returns `Jacq ues F ores t`. The helper counts the
tokens that have only 1 or 2 letters **and** are not a real short word (`LEGIT_SHORT` has *a, is, of, et, al* and
others). Fewer than 3 such tokens count as zero, so one stray `B.` is no problem.

In [ ]:
show(H._spaced_letter_ratio)

In [ ]:
def spaced_by_hand(text):
    words = text.split()
    bad = []
    for w in words:
        letters = re.sub(r"[^A-Za-z]", "", w)
        if letters and len(letters) <= 2 and letters.lower() not in H.LEGIT_SHORT:
            bad.append(w)
    ratio = 0.0 if len(bad) < 3 else len(bad) / len(words)
    return bad, ratio

for text in ["Jacq ues F ores t and M ari e wrote on it.",
             "It is up to us to do so in an ad hoc way.",
             "Study B and study C used the same scale."]:
    bad, ratio = spaced_by_hand(text)
    assert ratio == H._spaced_letter_ratio(text.split())
    verdict = "DROP" if ratio > H.MAX_SPACED else "keep"
    print(f"{text!r}\n    broken tokens {bad} -> ratio {ratio:.2f} -> {verdict} (limit {H.MAX_SPACED})")

### `_is_citation`: is the text inside a bracket a citation?

This helper decides what `strip_citations` removes. It receives the text **inside** `( )` or `[ ]` and applies
five rules in order. The first rule that fires decides:

In [ ]:
show(H._is_citation)

The traced copy below has the same five rules. It also returns **which rule decided**, so you can see why each
case comes out as it does:

In [ ]:
def is_citation_traced(inner):
    years = H.RE_YEAR.findall(inner)
    if not years:
        return False, "1: no year inside"
    if len(years) >= 2 and (";" in inner or "et al" in inner):
        return True, "2: two or more years, plus ';' or 'et al'"
    if len(re.findall(r"[A-Za-z]+", inner)) > 12:
        return False, "3: more than 12 words, so prose that mentions a year"
    if re.search(r"[A-Z][a-z]+|et al\.?|&|;", inner):
        return True, "4: has a capitalised name, 'et al', '&' or ';'"
    return inner.strip().isdigit(), "5: last rule: is it only digits (a bare year)?"

cases = ["Ryan & Deci, 2017",
         "Deci et al., 1999; Ryan, 2000",
         "2019",
         "see the 2017 review",
         "e.g., van der Kaap-Deeder, this volume",
         "i.e., introjection, external regulation",
         "a trend that began in the 1990s and continued in schools and workplaces in many countries since then"]
out = []
for inner in cases:
    verdict, why = is_citation_traced(inner)
    assert verdict == H._is_citation(inner)
    out.append({"inside the brackets": inner, "citation?": verdict, "decided by rule": why})
pd.DataFrame(out)

Note the `this volume` case. A handbook cites its own chapters without a year, so rule 1 keeps that text. You
will see `(e.g., Di Domenico & Ryan, this volume)` survive in the real sentences below.

### `_is_stat`: is the text inside a bracket a statistic?

It has three rules. The text must contain a statistics marker (`=`, `<`, `>`, `p <`, `n =`, `SD`, `CI`, `%`).
More than 15 % of its characters must be digits or symbols. It may have at most 4 words of 4 or more letters.
The last two rules keep prose like "(about 40% of the students ...)":

In [ ]:
show(H._is_stat)

In [ ]:
def is_stat_traced(inner):
    if not H.RE_STAT.search(inner):
        return False, "1: no statistics marker"
    nonalpha = sum(1 for c in inner if not c.isalpha() and not c.isspace())
    share = nonalpha / max(len(inner), 1)
    if share <= 0.15:
        return False, f"2: only {share:.0%} digits/symbols (limit 15%)"
    long_words = len(re.findall(r"[A-Za-z]{4,}", inner))
    if long_words <= 4:
        return True, f"3: {share:.0%} digits/symbols and {long_words} long words (limit 4)"
    return False, f"3: {long_words} long words (limit 4), so prose with numbers"

cases = ["p < .05, n = 120",
         "r = .45, 95% CI [.30, .58]",
         "i.e., introjection, external regulation",
         "roughly 40% of the students in the sample reported feeling controlled",
         "M = 3.2, SD = 0.8, for autonomous regulation among adolescents and adults"]
out = []
for inner in cases:
    verdict, why = is_stat_traced(inner)
    assert verdict == H._is_stat(inner)
    out.append({"inside the brackets": inner, "statistic?": verdict, "decided by rule": why})
pd.DataFrame(out)

### `_strip_noise` and `_tidy`: remove the brackets, then repair the gap

`_strip_noise(text, is_noise)` is shared by `strip_citations` and `strip_stats`. It contains a small function,
`keep_or_cut`, which `re.sub` calls once for each bracket group it finds. `match.group(1)` is the text inside the
brackets and `match.group(0)` is the whole group with its brackets. The function returns `""` to cut the group
and `group(0)` to keep it. `_tidy` then repairs the spaces and punctuation that the cut leaves behind.

In [ ]:
show(H._strip_noise)
show(H._tidy)

In [ ]:
sentence = "Autonomy predicts persistence (Ryan & Deci, 2017), and effort (p < .05) [see Table 2]."

print("groups found by RE_PAREN :", H.RE_PAREN.findall(sentence))
print("groups found by RE_BRACK :", H.RE_BRACK.findall(sentence))
print()
cut = H.RE_PAREN.sub(lambda m: "" if H._is_citation(m.group(1)) else m.group(0), sentence)
cut = H.RE_BRACK.sub(lambda m: "" if H._is_citation(m.group(1)) else m.group(0), cut)
print("citations cut, before _tidy:", repr(cut))

t = re.sub(r"\s+([,.;:])", r"\1", cut);       print("  _tidy 1, no space before , . ; :   ", repr(t))
t = re.sub(r"\(\s*\)|\[\s*\]", "", t);         print("  _tidy 2, remove empty () and []    ", repr(t))
t = re.sub(r"[,;]\s*([.!?])", r"\1", t);       print("  _tidy 3, ',.' becomes '.'          ", repr(t))
t = re.sub(r"\s+", " ", t).strip();            print("  _tidy 4, one space between words   ", repr(t))

assert t == H._strip_noise(sentence, H._is_citation)
print()
print("then strip_stats on that:", repr(H._strip_noise(t, H._is_stat)))
print("[see Table 2] stays: it has no year (no citation) and no statistics marker (no statistic).")

`_tidy` rule 1 also removed the space in `(p < .05)`, which became `(p <.05)`, because it deletes any space before
a `.`. This does no harm here, because `strip_stats` removes the whole group next.

### `_page_list`: turn a page selection into a set of page numbers

Pages count from 1, as in a PDF viewer. The helper accepts an int, a list, or a string like `"12-14,20"`.

In [ ]:
show(H._page_list)
for spec in [7, [3, 5, 9], "12-14,20", " 1-3 , 8 "]:
    print(f"{spec!r:<14} -> {sorted(H._page_list(spec))}")

## 1.4 A small made-up book that triggers every rule

Before we use a real paper, we write three pages of markdown by hand. This is the kind of text that
`pdf_inspector` returns: one paragraph per line, a blank line between paragraphs, `##` for headings.
Each odd line is there to trigger one rule. The comment at the end of each line says which rule we expect.

In [ ]:
LONG = "When teachers listen to students and explain their requests, " * 9 + "motivation grows."

toy_pages = {
    1: "\n".join([
        "The Oxford Handbook of Motivation",                                     # running head (3 pages)
        "",
        "## Autonomy and Motivation",                                            # heading -> section name
        "",
        "When teachers support **autonomy**, students show more intrinsic motivation (Ryan & Deci, 2017). "
        "Controlling rewards reduce interest in the task (p < .05, n = 120), and this effect holds across "
        "[several studies](https://example.org).",                              # citation, stat, markdown
        "",
        "The eﬀect of teachers’ “autonomy support” is signiﬁcant – and lasting.",  # unicode
        "",
        "This interest in inte-",                                                # hyphen at a line end
        "grative knowledge grows with age, as the inte- grative view predicts.", # hyphen inside a line
        "",
        "| Study | N | Effect |",                                                # table
        "| --- | --- | --- |",
        "",
        "12",                                                                    # page number
    ]),
    2: "\n".join([
        "The Oxford Handbook of Motivation",
        "",
        "Figure 1. The self-determination continuum of regulation types.",      # caption
        "",
        "SELF-DETERMINATION THEORY IN PRACTICE TODAY.",                          # no lowercase
        "",
        "Jacq ues F ores t and M ari e wrote on this topic in detail.",          # spaced letters
        "",
        "The scores were 12, 15, 19, 22, 31 and 40 in 2019.",                    # digit heavy
        "",
        "See also intrinsic motivation and autonomy support.",                   # index entry
        "",
        "Why do external rewards backfire?",
        "",
        "Need frustration predicts ill-being and, over time, it",                # cut by the page break
    ]),
    3: "\n".join([
        "The Oxford Handbook of Motivation",
        "",
        "leads to defiance and disengagement among adolescents.",                # rest of the cut sentence
        "",
        "3.14 / 2.71 = 1.16 + 0.5",                                              # low alpha line
        "",
        "Key words: autonomy, competence, relatedness",                          # keywords label
        "",
        "https://doi.org/10.1037/0003-066X.55.1.68",                             # DOI
        "",
        "pp.123-145",                                                            # reference fragment
        "",
        "Autonomy support as a resource for teachers",                          # no final punctuation
        "",
        "OK then.",                                                              # too short
        "",
        LONG,                                                                    # 83 words
        "",
        "## References",                                                         # start of reference section
        "",
        "Deci, E. L., & Ryan, R. M. (2000). The what and why of goal pursuits. Psychological Inquiry, 11, 227-268.",
        "",
        "CHAPTER",                                                               # ends the reference section
        "",
        "Contents",                                                              # front matter
        "",
        "A new chapter begins with a sentence about competence.",
    ]),
}
print(sum(len(p.split(chr(10))) for p in toy_pages.values()), "lines on", len(toy_pages), "pages;",
      "the long sentence has", len(LONG.split()), "words")

## 1.5 The ten steps, run by hand on the made-up book

**Step 3 of `harvest`: one row per line.** Blank lines are rows too, because a blank line is what separates
two paragraphs.

In [ ]:
rows = []
for page in sorted(toy_pages):
    for line in toy_pages[page].split("\n"):
        rows.append({"doc_id": "toy", "page": page, "section": "",
                     "text": line, "kept": True, "drop_reason": "", "flag": ""})
print(len(rows), "rows")
pd.DataFrame(rows).head(8)

**Important: the steps change the row dicts in place.** `fix_unicode` writes `r["text"] = ...` into the same
dict it received. To see what a step changed, we keep a deep copy of the rows from before the step and compare.
`changes()` prints only the rows that changed.

In [ ]:
def changes(before, after):
    # Show rows whose text, kept, drop_reason or flag changed. The same list length is expected.
    out = []
    for i, (b, a) in enumerate(zip(before, after)):
        diff = {}
        if b["text"] != a["text"]:
            diff["text before"] = b["text"]
            diff["text after"] = a["text"]
        if b["kept"] != a["kept"]:
            diff["drop_reason"] = a["drop_reason"]
        if b.get("flag") != a.get("flag"):
            diff["flag"] = a["flag"]
        if diff:
            out.append({"row": i, "page": a["page"], **diff})
    print(len(out), "rows changed")
    return pd.DataFrame(out).fillna("")

### Step 1 `fix_unicode`

In [ ]:
show(H.fix_unicode)
before = copy.deepcopy(rows)
rows = H.fix_unicode(rows)
changes(before, rows)

### Step 2 `strip_markdown`

Two regexes, and each one keeps the words. `RE_MDLINK` changes `[text](url)` to `text`. Its group 1 is
the link text and the URL is thrown away. `RE_EMPH` changes `**word**` / `*word*` / `_word_` to `word`. The
`\1` in that pattern requires the same marker at both ends, and the lookarounds `(?=\S)` / `(?<=\S)` require a
non-space character just inside the markers, so `3 * 4 * 5` is left alone.

In [ ]:
show(H.strip_markdown)
before = copy.deepcopy(rows)
rows = H.strip_markdown(rows)
changes(before, rows)

### Step 3 `drop_running_heads`

This step makes two passes over the rows:

1. Count every non-empty line of at most 90 characters.
2. Drop every line that occurs 3 or more times. A running head repeats on every page, and a sentence of prose
   almost never repeats word for word.

In [ ]:
show(H.drop_running_heads)
counts = collections.Counter(r["text"].strip() for r in rows if 0 < len(r["text"].strip()) <= 90)
print("lines seen 2+ times:", {line: n for line, n in counts.items() if n >= 2})
before = copy.deepcopy(rows)
rows = H.drop_running_heads(rows)
changes(before, rows)

### Step 4 `drop_junk_lines`

This step reads the lines from top to bottom and keeps two things in memory:

- `section`: the text of the last `##` heading. Every kept line below that heading gets it as `r["section"]`.
  The heading line itself is dropped.
- `in_refs`: `True` from a `## References` / `Notes` / `Acknowledgements` heading until the next `CHAPTER` or
  `PART` line. Everything in between is dropped as `reference_section`.

Every other line goes through an `if/elif` chain. The **first** rule that fires sets the drop reason. The last
rule, `_alpha_ratio(line) < 0.55`, catches lines that are mostly symbols or digits.

In [ ]:
show(H.drop_junk_lines)

In [ ]:
before = copy.deepcopy(rows)
rows = H.drop_junk_lines(rows)
changes(before, rows)

`blank` rows are dropped too. They are not junk, but a dropped row is what **closes a paragraph** in the next
step. The reference entry `Deci, E. L., & Ryan ...` would have passed every other rule, so it is the
`in_refs` memory that drops it. The line after `CHAPTER` is kept again.

The captions (`Figure 1.`), the keywords, the all-caps line and the spaced letters are all still kept here.
They are dropped at sentence level in step 9.

### Step 5 `join_paragraphs`

This is the only step that changes the **number** of rows. It works in two loops.

**Loop 1, lines to paragraphs.** It walks the rows and keeps an open paragraph, `block`:
- a dropped row: keep it for the log and **close** the paragraph (`block = None`);
- a kept row and no open paragraph: start a new paragraph (page and section come from this first line);
- a kept row after a line that ends in `-`, and this row starts lowercase: join them **without** the hyphen
  (`inte-` + `grative` → `integrative`);
- any other kept row: join it with a space.

**Loop 2, repair each paragraph.**
- `(\w)- ([a-z])` → `\1\2` joins a word that was split *inside* a line (`inte- grative` → `integrative`).
- If the previous paragraph has **no final punctuation** and this one **starts lowercase**, a page break cut a
  sentence, so the two are merged.

The function returns `paragraphs + dropped rows`. The dropped rows move to the end but stay in the list.

In [ ]:
show(H.join_paragraphs)

In [ ]:
n_before = len(rows)
rows = H.join_paragraphs(rows)
paras = [r for r in rows if r["kept"]]
print(f"{n_before} line rows -> {len(paras)} paragraphs + {len(rows) - len(paras)} dropped rows kept for the log")
pd.DataFrame(paras)[["page", "section", "text"]]

Check the paragraphs:

- `integrative` is repaired in **both** places: the hyphen at the end of a line, and the `inte- grative` inside a line.
- *"Need frustration predicts ill-being and, over time, it"* (page 2) and *"leads to defiance ..."* (page 3)
  are one paragraph again. The page-3 running head between them was dropped, so it closed the paragraph, and the
  merge rule joined the two parts again.
- A paragraph keeps the **page and section of its first line**.

### Step 6 `split_sentences`

Each paragraph is split with `SPLITTER`. Each sentence becomes a **new** row with two new keys:

- `sent_id = f"p{i:04d}s{j:03d}"`: `i` is the paragraph's position in the list, `j` is the sentence's position in
  the paragraph. `p0003s001` is the 2nd sentence of the 4th paragraph. Every later file and relation can be
  traced back to its sentence through this id.
- `raw`: a copy of the sentence at this moment, **before** citations and statistics are removed.
  `text` is changed from here on, and `raw` stays as it is.

In [ ]:
show(H.split_sentences)
rows = H.split_sentences(rows)
sents = [r for r in rows if r["kept"] and "sent_id" in r]
pd.DataFrame(sents)[["sent_id", "page", "section", "text"]]

### Steps 7 and 8 `strip_citations`, `strip_stats`

Both steps apply `_strip_noise` (section 1.3) to `text`. `raw` keeps the original sentence.

In [ ]:
show(H.strip_citations)
before = copy.deepcopy(rows)
rows = H.strip_citations(rows)
rows = H.strip_stats(rows)
changes(before, rows)

### Step 9 `drop_junk_sentences`

The same `if/elif` pattern, now per sentence. The first six rules **drop**. The last two only **flag** and the
sentence is kept. This follows the package rule "drop structural junk only, flag the doubtful".

In [ ]:
show(H.drop_junk_sentences)

In [ ]:
before = copy.deepcopy(rows)
rows = H.drop_junk_sentences(rows)
changes(before, rows)

Two results to note:

- The splitter cut `Figure 1. The self-determination ...` into `Figure 1.` and the rest. `Figure 1.` is dropped as
  a caption. The rest of the caption stays, because on its own it reads like a sentence.
- The 83-word sentence got the flag `long` and is kept, **for now**. Step 10 follows.

### Step 10 `filter_length`

This step has one rule: drop a sentence with fewer than `min_words` or more than `max_words` words. It counts
words in `text`, which is the text *after* citations were removed.

In [ ]:
show(H.filter_length)
before = copy.deepcopy(rows)
rows = H.filter_length(rows)
changes(before, rows)

The long sentence is dropped here, with the flag `long` still on it. The default `max_words=80` is the same
number as `LONG_FLAG = 80`, so **with the defaults every sentence flagged `long` is then dropped as `too_long`**.
The flag never reaches the output. (Section 4.2 comes back to this.)

**The end of `harvest`: split and reshape.** The code keeps the rows with `kept=True` that have a
`sent_id`, and renames `text` to `clean`. The drop log is every row with `kept=False` except the blank rows:

In [ ]:
kept_by_hand = [r for r in rows if r["kept"] and "sent_id" in r]
dropped_by_hand = [r for r in rows if not r["kept"] and r["drop_reason"] != "blank"]
print(len(kept_by_hand), "sentences kept |", len(dropped_by_hand), "rows in the drop log")
pd.DataFrame(kept_by_hand)[["sent_id", "page", "raw", "text", "flag"]].rename(columns={"text": "clean"})

## 1.6 The real `harvest()` on the same made-up book

Now the source of `harvest` itself, and of `_harvest_one`, which does the work for one PDF.
`_harvest_one` calls the four helpers listed at the start of Part 1. Then it puts the document number in front
of each `sent_id`: `d000p0003s001` is document 0, paragraph 3, sentence 1.

In [ ]:
show(H.harvest)
show(H._harvest_one)

**The cache.** `_read_pages` first looks for `<pdf>.pages.json`. If that file exists, it **never opens the PDF**.
`harvest` also accepts a PDF path when only its cache file exists. So we can run the real function on our made-up
book: write `toy_pages` to `toy.pdf.pages.json` in a temporary folder and ask for `toy.pdf`, a file that does not
exist.

The page cache is the only file that `harvest` writes. (It is gitignored as `*.pages.json`.)

In [ ]:
tmp = tempfile.mkdtemp()
fake_pdf = os.path.join(tmp, "toy.pdf")
with open(fake_pdf + ".pages.json", "w", encoding="utf-8") as fh:
    json.dump(toy_pages, fh)              # json turns the int keys into "1", "2", "3"; harvest turns them back

toy = harvest(fake_pdf)

assert [s["clean"] for s in toy["sentences"]] == [r["text"] for r in kept_by_hand]
assert [d["drop_reason"] for d in toy["dropped"]] == [r["drop_reason"] for r in dropped_by_hand]
print("\nThe real function gives exactly the result of the ten steps run by hand.")

**The knobs.** `pages` and `skip_pages` choose pages before anything else happens. `steps`
replaces the list of steps. A step can be the name of a built-in step, or **your own function** with the same
contract (a list of rows in, a list of rows out, mark a row instead of deleting it):

In [ ]:
def drop_questions(rows):
    # A custom step: drop sentences that are questions. It keeps the row and marks it.
    for r in rows:
        if r["kept"] and "sent_id" in r and r["text"].rstrip().endswith("?"):
            r["kept"] = False
            r["drop_reason"] = "question"
    return rows

print("--- pages 1-2 only, and page 2 skipped ---")
only_p1 = harvest(fake_pdf, pages="1-2", skip_pages=2)
print("\n--- default steps plus a custom step at the end ---")
custom = harvest(fake_pdf, steps=H.DEFAULT_STEPS + [drop_questions])
print("\n--- without strip_citations ---")
no_cite = harvest(fake_pdf, steps=[s for s in H.DEFAULT_STEPS if s != "strip_citations"])
print("\nfirst sentence now:", no_cite["sentences"][0]["clean"])

## 1.7 `harvest()` on a real paper, and on a folder

The same call on three pages of a real paper from `sample_files/`. The first call on a PDF extracts all
pages and caches them. That takes about a second. After that the cache is used.

In [ ]:
PDF = "sample_files/schutz_2025_loneliness_social_support.pdf"
book = harvest(PDF, pages="2-4")

In [ ]:
book_df = pd.DataFrame(book["sentences"])
print("columns:", list(book_df.columns))
book_df[["sent_id", "page", "section", "clean", "flag"]].head(8)

Where `raw` and `clean` differ, `strip_citations` / `strip_stats` removed something:

In [ ]:
changed = book_df[book_df.raw != book_df.clean]
print(f"{len(changed)} of {len(book_df)} sentences lost a citation or a statistic")
changed[["sent_id", "raw", "clean"]].head(6)

In [ ]:
pd.DataFrame(book["dropped"])[["page", "drop_reason", "text"]]

**A folder.** Give a folder, and `harvest` reads every PDF in it. `_harvest_folder` reads the PDFs one at a
time, so two papers never share a paragraph or a running head. A bad file does not stop the run: it gets one row
in the drop log with `page` 0 and the reason `not_pdf` or `unreadable_pdf`.

In [ ]:
show(H._harvest_folder)
folder = harvest("sample_files")
pd.DataFrame([r for r in folder["dropped"] if r["page"] == 0])

---
# Part 2 · `causenet.py`: from sentences to cause → effect pairs

`causenet(sentences)` wraps the fine-tuned model `rasoultilburg/SocioCausaNet` (Paper 1 of the series). For each
sentence the model decides **is this sentence causal?** It then marks **cause spans** and **effect spans** and
links them in pairs. `causenet` itself does three things:

1. load the model once (`_load_model`);
2. run it in batches (`model.predict`);
3. flatten the output into one **relation dict** per cause → effect pair, with the spans repaired by `tidy_span`.

## 2.1 `_load_model`: load once, keep in memory

`_MODEL`, `_TOKENIZER` and `_DEVICE` are module-level variables. The first call fills them. Every later call sees
`_MODEL is not None` and returns at once. `trust_remote_code=True` is needed because the model's architecture
(`modeling_joint_causal.py`) lives in the Hub repository, not in `transformers`. `.eval()` switches off dropout.

In [ ]:
show(C._load_model)

In [ ]:
import transformers
transformers.utils.logging.disable_progress_bar()

t0 = time.time(); model, tokenizer = C._load_model(); print(f"first call : {time.time() - t0:.1f}s")
t0 = time.time(); model, tokenizer = C._load_model(); print(f"second call: {time.time() - t0:.4f}s (already in memory)")
print(type(model).__name__, "on", C._DEVICE)

## 2.2 The input: 60 sentences of one real paper

For the rest of the notebook we use the first 60 kept sentences of the paper from section 1.7. `harvest`
already cleaned them. Each one is a sentence row with `clean`, `sent_id`, `doc_id` and `page`.

In [ ]:
sentences = book["sentences"][:60]
print(len(sentences), "sentences")
pd.DataFrame(sentences)[["sent_id", "page", "clean"]].head(4)

## 2.3 What the model returns before `causenet` touches it

`model.predict` takes a list of strings and returns one dict per sentence. We call it directly on three sentences:
one that is not causal, one simple causal sentence, and a real one from the paper.

In [ ]:
demo = ["The weather in Tilburg was grey that morning.",
        "Autonomy support increases intrinsic motivation among students.",
        sentences[1]["clean"]]
raw_preds = model.predict(demo, tokenizer=tokenizer, rel_mode="neural",
                          rel_threshold=0.8, cause_decision="cls+span")
for sentence, p in zip(demo, raw_preds):
    print("SENTENCE:", sentence)
    print("  keys   :", list(p.keys()))
    print("  causal :", p["causal"])
    for rel in p["relations"]:
        print(f"  relation: {rel['cause']!r:<30} -> {rel['effect']!r:<32} ({rel['type']})")
    print()

What to notice:

- **`causal`** is the sentence-level verdict. A non-causal sentence has an empty `relations` list. (It also carries
  an empty `spans` key, which `causenet` never reads.)
- **`relations`** is a list of `{cause, effect, type}`. `causenet` uses only `causal` and `relations`.
- In the long sentence the model **crosses every cause with every effect** (2 causes × 5 effects = 10 relations).
  Some effects are single words cut from a list (`"positive"`, `"persistence,"`).
- Spans come back **lowercased** and sometimes with a **trailing comma**. The span is decoded from token ids, not
  cut out of the sentence, so tokenizer debris can appear in it. `tidy_span` (2.5) repairs this.

## 2.4 The knobs: `threshold` and `decision`

- `threshold` (the model's `rel_threshold`): the confidence needed to accept a cause→effect link.
- `decision` (the model's `cause_decision`): what makes a sentence causal.
  `cls+span` needs both the sentence classifier **and** at least one tagged span. `span_only` needs only a span.

The loop runs the grid that the project uses on the 60 sentences and counts the results:

In [ ]:
texts = [s["clean"] for s in sentences]
grid = []
for decision in ["cls+span", "span_only"]:
    for threshold in [0.5, 0.6, 0.7, 0.8]:
        preds = model.predict(texts, tokenizer=tokenizer, rel_mode="neural",
                              rel_threshold=threshold, cause_decision=decision)
        grid.append({"decision": decision, "threshold": threshold,
                     "causal sentences": sum(p["causal"] for p in preds),
                     "relations": sum(len(p["relations"]) for p in preds)})
pd.DataFrame(grid)

On these sentences the two `decision` rules give the same result: every sentence where the model tagged a span
was also classified as causal. The threshold changes only the **number of links** among those sentences, because
a higher threshold drops the weaker cause→effect pairs. The same sentences stay causal.

## 2.5 `tidy_span`: repair the decoding and keep every word

It lives in `harmonizer/spans.py` because both `causenet` and the harmonizer use it. It is the **only** change the
package ever makes to a span. The project rule is that nothing may rewrite a sentence or span after the model has
seen it, because the evaluation traces every span back to its sentence. So `tidy_span` removes **no word**. It
only undoes what the tokenizer broke.

In [ ]:
show(S.tidy_span)

The traced copy runs the same lines in order and prints the string after each line. A `•` marks the lines that
changed something:

In [ ]:
def tidy_span_traced(text):
    trace = [("input", text)]
    t = S._SPECIAL.sub(" ", text);                   trace.append(("[CLS]/[SEP]/... -> space", t))
    t = re.sub(r"\s+##", "", t);                      trace.append(("join ' ##piece' to its word", t))
    t = t.replace("##", "");                          trace.append(("drop a leading ##", t))
    t = re.sub(r"\s+'\s*", "'", t);                   trace.append(("society ' s -> society's", t))
    t = re.sub(r"\s+([,.;:!?%)\]}])", r"\1", t);      trace.append(("no space before ) , .", t))
    t = re.sub(r"([(\[{])\s+", r"\1", t);             trace.append(("no space after (", t))
    t = re.sub(r"\s+", " ", t);                       trace.append(("one space", t))
    t = t.lstrip(S._EDGE_LEFT).rstrip(S._EDGE_RIGHT); trace.append(("strip edge punctuation", t))
    if t.count("(") != t.count(")"):
        t = t.replace("(", " ").replace(")", " ")
        t = re.sub(r"\s+", " ", t).strip(S._EDGE_LEFT + S._EDGE_RIGHT)
        trace.append(("unbalanced bracket -> remove it", t))
    previous = None
    for name, value in trace:
        mark = "•" if previous is not None and value != previous else " "
        print(f"   {mark} {name:<34} {value!r}")
        previous = value
    return t

for broken in ["[CLS] autonomy support is critical [SEP]",
               "inte ##grative processes",
               "society ' s desired values",
               "persistence,",
               "people ' s ( mal ) adaptative functioning",
               "), and intrapersonal events"]:
    print(repr(broken))
    assert tidy_span_traced(broken) == S.tidy_span(broken)
    print()

**Why the input must pass `fix_unicode` first.** The apostrophe rule matches only the straight `'`. Many
PDFs use the curly `’`. `harvest` changes `’` to `'` in step 1, so a harvested sentence is safe. If you give
`causenet` raw strings with curly quotes, the space around the apostrophe stays:

In [ ]:
print(repr(S.tidy_span("people ’ s ( mal ) adaptative functioning")), " <- curly quote, not repaired")
print(repr(S.tidy_span("people ' s ( mal ) adaptative functioning")), "       <- after fix_unicode")

## 2.6 `causenet()`, line by line

1. **Accept plain strings.** A string becomes a row with `sent_id = f"s{i:04d}"` and `doc_id = "input"`, so
   `causenet(["some sentence"])` works without `harvest`.
2. **Batches.** It sends `batch_size` sentences to `model.predict` at a time and prints the progress on one line
   (`\r`).
3. **Flatten.** `zip(rows, preds)` pairs each sentence with its prediction. Non-causal sentences are skipped.
   Each relation `k` becomes one dict with `rel_id = sent_id + f"r{k:02d}"`. That id is the link back to the
   sentence.
4. **Two counts** that check the rule "spans stay substrings of their sentence":
   `n_tidied` counts relations where `tidy_span` changed something. `n_verbatim` counts relations whose cause and
   effect **both** occur in the lowercased sentence.

In [ ]:
show(C.causenet)

In [ ]:
relations = causenet(sentences)
rel_df = pd.DataFrame(relations)
rel_df[["rel_id", "cause", "effect"]].head(12)

How to read one `rel_id`: `d000p0001s000r03` = document 0, paragraph 1 (counted from 0), sentence 0,
relation 3. The full sentence
and the page travel with every relation:

In [ ]:
relations[3]

**The relations that are not found verbatim** in their sentence, and the reason:

In [ ]:
not_verbatim = [r for r in relations
                if not (r["cause"].lower() in r["sentence"].lower() and r["effect"].lower() in r["sentence"].lower())]
print(f"{len(relations) - len(not_verbatim)} of {len(relations)} relations are verbatim")
for r in not_verbatim[:6]:
    missing = [k for k in ("cause", "effect") if r[k].lower() not in r["sentence"].lower()]
    print(f"\n{r['rel_id']}  missing: {missing}")
    for k in missing:
        print(f"   {k}: {r[k]!r}")
    print(f"   sentence: {r['sentence'][:160]!r}")

In each of these cases the tokenizer **lost or added a space**, and `tidy_span` cannot tell which space was real:

- `and/or` is decoded as `and / or`. `tidy_span` has no rule for `/`.
- `(mal)adaptative` is decoded as `( mal ) adaptative`. The bracket rules repair the inside, but the space
  after `)` stays, because in most spans that space is real.
- `humans' integrative` is decoded as `humans ' integrative`. The apostrophe rule joins both sides, which is right
  for `society ' s` but wrong for a plural possessive, so the space after the apostrophe is lost.

The words are all there. Only the spacing differs, so a comparison that ignores spaces would find these spans.

---
# Part 3 · `harmonizer/`: from raw spans to named constructs

`causenet` returns raw spans, in the author's own words. The same construct appears as *"intrinsic
motivation"*, *"intrinsic motivation among students"*, *"autonomous motivation"* and so on. The harmonizer
brings the spans together in two ways, and both rest on one idea: **a text becomes a vector, and similar texts
have vectors that point the same way**.

| file | what it holds |
|---|---|
| `embeddings.py` | `embed()`: text → unit vector; the only place a sentence-transformer is loaded |
| `spans.py` | `tidy_span()` (seen in 2.5) and `collect_spans()` |
| `taxonomy.py` | read ELSST, one entry per concept path, embed them once, cache to disk |
| `standardize.py` | span → nearest taxonomy concepts; group → nearest taxonomy concepts |
| `pairwise.py` | spans → groups by a cosine threshold (connected components) |
| `cluster.py` | spans → groups by density (HDBSCAN) |
| `labels.py` | group → a label made from its own words (no taxonomy) |

## 3.1 `embed`: text to unit vector

In [ ]:
show(E.embed)
print("SHORTHAND =", E.SHORTHAND)

Line by line:
- `SHORTHAND.get(model, model)` turns `"allmpnet"` into the full Hugging Face id. A full id passes through unchanged.
- `_LOADED` is a dict that stays in memory: each model loads **once** per session.
- `normalize_embeddings=True` scales every vector to **length 1**. For vectors of length 1 the cosine similarity
  is the plain dot product, so the whole package computes similarity as `A @ B.T`.

In [ ]:
toy_spans = ["intrinsic motivation", "intrinsic motivation among students", "autonomous motivation",
             "well-being", "wellness", "psychological well-being",
             "controlling rewards", "external rewards"]
V = E.embed(toy_spans)
print("shape:", V.shape, "| length of each vector:", np.linalg.norm(V, axis=1).round(4))

a, b = V[0], V[2]
print("cosine by the formula a.b/(|a||b|):", round(float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))), 4))
print("dot product a.b                   :", round(float(a @ b), 4))

In [ ]:
sim = V @ V.T                     # every pair at once: (8, 768) @ (768, 8) -> (8, 8)
pd.DataFrame(sim, index=toy_spans, columns=toy_spans).round(2)

## 3.2 `collect_spans`: the list of unique spans

Every harmonizer function starts with this function. It accepts **relations** (it takes `cause` and `effect`
from each) or **plain strings**, strips the spaces at both ends, drops duplicates, and keeps the order in which
it first saw each span. The span string itself, unchanged, is the key of every result dict later. That is how a
result traces back to its relation.

In [ ]:
show(S.collect_spans)
print(S.collect_spans(relations[:4]))
print(S.collect_spans(["  autonomy ", "autonomy", "competence", "autonomy"]))
spans = S.collect_spans(relations)
print(f"\n{len(relations)} relations -> {len(spans)} unique spans")

## 3.3 `taxonomy.py`: a thesaurus as a list of vectors

Standardisation compares each span with every concept of a thesaurus. `taxonomy.py` prepares that side. It
**parses** the file, turns each concept into **entries**, chooses a **text** for each entry, **embeds** the texts
and **caches** the vectors.

### Parsing ELSST: one `<rdf:Description>` block per concept

The loader does not use an RDF library. It uses regular expressions. First, here is what one concept looks like
in the raw file. The code keeps the English lines and the structural lines of the `SELF-ESTEEM` block:

In [ ]:
print("files:", T.DEFAULT_FILES)
elsst_text = open(T.DEFAULT_FILES["elsst"], encoding="utf-8").read()
blocks = re.findall(r'<rdf:Description[^>]*rdf:about="([^"]+)"(.*?)</rdf:Description>', elsst_text, re.S)
print(f"{len(elsst_text):,} characters, {len(blocks):,} <rdf:Description> blocks")

uri, body = next((u, b) for u, b in blocks if '<skos:prefLabel xml:lang="en">SELF-ESTEEM<' in b)
print("\nrdf:about =", uri)
for line in body.splitlines():
    if 'xml:lang="en"' in line or any(k in line for k in ("broader", "narrower", "topConceptOf", "rdf:type")):
        print("  ", line.strip())

`_load_elsst` reads each block with six small regexes (`prefLabel`, `altLabel`, `scopeNote`, `definition`,
`broader`, `narrower`) and builds **one concept dict** per block. `is_leaf` means "has no narrower concept".
`is_top` means "is a top concept of the scheme". The same loop as in the loader, run by hand:

In [ ]:
show(T._load_elsst)

In [ ]:
import html
re_pref = re.compile(r'<skos:prefLabel xml:lang="en">(.*?)</skos:prefLabel>', re.S)
re_alt = re.compile(r'<skos:altLabel xml:lang="en">(.*?)</skos:altLabel>', re.S)
re_note = re.compile(r'<skos:scopeNote xml:lang="en">(.*?)</skos:scopeNote>', re.S)
re_def = re.compile(r'<skos:definition xml:lang="en">(.*?)</skos:definition>', re.S)
re_broader = re.compile(r'<skos:broader rdf:resource="([^"]+)"')
re_narrower = re.compile(r'<skos:narrower rdf:resource="([^"]+)"')

concepts = {}
for u, b in blocks:
    if "skos/core#Concept" not in b:
        continue
    pref = re_pref.search(b)
    if not pref:
        continue
    note = re_note.search(b) or re_def.search(b)
    concepts[u] = {"label": html.unescape(pref.group(1)).strip(),
                   "alt": [html.unescape(a).strip() for a in re_alt.findall(b)],
                   "note": html.unescape(note.group(1)).strip() if note else "",
                   "broader": re_broader.findall(b),
                   "is_leaf": not re_narrower.findall(b),
                   "is_top": "skos:topConceptOf" in b}
print(f"{len(concepts):,} concepts")
concepts[uri]

### From concepts to entries: every root-to-concept path (`_entries`)

`SELF-ESTEEM` has **two** broader concepts. This is poly-hierarchy: the concept sits in two places in the tree.
`paths_to_root` climbs through each parent recursively until it reaches a top concept, and returns **every** path.
The package keeps all paths, never only the shortest, because each path gives the concept a different context
when it is embedded. The traced copy prints the climb:

In [ ]:
show(T._entries)

In [ ]:
def paths_to_root_traced(u, seen, depth=0):
    c = concepts[u]
    parents = [p for p in c["broader"] if p in concepts and p not in seen]
    print("   " * depth + f"{c['label']}  (top={c['is_top']}, parents={[concepts[p]['label'] for p in parents]})")
    if c["is_top"] or not parents:
        return [[u]]
    paths = []
    for p in parents:
        for above in paths_to_root_traced(p, seen | {u}, depth + 1):
            paths.append(above + [u])
    return paths

paths = paths_to_root_traced(uri, set())
print()
for p in paths:
    print(" > ".join(concepts[x]["label"] for x in p))

In [ ]:
elsst_entries = T.load_taxonomy("elsst", leaves_only=True)
mine = [e for e in elsst_entries if e["id"] == uri]
assert [e["path"] for e in mine] == [" > ".join(concepts[x]["label"] for x in p) for p in paths]
mine

One entry per path, each with the same `id`. Note that `parents` lists the **nearest parent first**. The count line
shows the effect of `leaves_only`. With `True` only leaf concepts become entries. With `False` every intermediate
concept (`MOTIVATION`, `PERSONAL IDENTITY`, ...) can also be a label. This is a choice of granularity, not of
accuracy.

In [ ]:
all_entries = T.load_taxonomy("elsst", leaves_only=False)

### What text stands for an entry: `strategy_text`

The embedding model sees one string per entry, not the entry dict. The strategy decides what that string is:
the leaf alone, the path, the leaf with its parents, and so on. `enriched` (the default) puts everything in: leaf,
alternative labels, scope note, path.

In [ ]:
show(T.strategy_text)

In [ ]:
for entry in [mine[0], [e for e in apa_entries if e["leaf"] == "Intrinsic Motivation"][0]]:
    print(entry["path"])
    for strategy in T.STRATEGIES:
        print(f"   {strategy:<9} {T.strategy_text(entry, strategy)!r}")
    print()

### Embedding and caching: `taxonomy_embeddings`

1. Load the entries (from memory if this session already parsed the file).
2. Build the cache file name from four settings: `<tax>_<strategy>_<model>_<scope>.npy`.
3. If the file exists **and** has one row per entry, load it and return. The row check catches a stale cache.
4. Otherwise embed `strategy_text(e, strategy)` for every entry, save the vectors as float32 `.npy`, and save the
   entries as `.json` in the same row order. Row `i` of the matrix is entry `i`.

The first run here embeds the 2,661 ELSST leaf entries. Later runs load the file.

In [ ]:
show(T.taxonomy_embeddings)

In [ ]:
entries, tax_vectors = T.taxonomy_embeddings("elsst", strategy="enriched", model="allmpnet", leaves_only=True)
print("entries:", len(entries), "| vectors:", tax_vectors.shape, tax_vectors.dtype)
print("cache folder:", sorted(os.listdir(T.CACHE_DIR)))

## 3.4 `standardize_constructs`: each span → its nearest taxonomy concepts

This is the whole function:

1. `spans = collect_spans(items)`.
2. Get the taxonomy vectors (cached).
3. Embed the **query text** of each span. `_queries` = `tidy_span(span).lower()`. The key stays the original span.
4. `scores = span_vectors @ tax_vectors.T`: one row per span, one column per entry, each cell a cosine. It is
   computed in blocks of 2,000 spans so the matrix stays small on a whole book.
5. For each row, `_top_matches` takes the best entries, **one per concept**.

No threshold is applied. Every span gets its top-k concepts, even when the best one is far away. The later graph
stage decides what counts as "standardised".

In [ ]:
show(St._queries)
show(St._top_matches)

**Why `_top_matches` skips `seen` ids.** A concept with two paths is **two rows** of the matrix. Both rows can
land near the top. The loop walks the entries from best to worst score (`np.argsort(-scores)`) and skips a concept
id that it has already taken. We check this with a query that we know lands on `SELF-ESTEEM`:

In [ ]:
q = E.embed(St._queries(["low self-esteem"]))[0]
scores_q = tax_vectors @ q
order = np.argsort(-scores_q)[:6]
pd.DataFrame([{"rank": k, "matrix row": int(j), "concept id (end)": entries[j]["id"][-12:],
               "path": entries[j]["path"], "score": round(float(scores_q[j]), 4)}
              for k, j in enumerate(order)])

In [ ]:
pd.DataFrame(St._top_matches(scores_q, entries, top=4))

The two `SELF-ESTEEM` rows collapse into one match, and the 4 slots go to 4 **different** concepts.

Now the whole function **by hand** on all our spans, then the real call, then the check:

In [ ]:
span_vectors = E.embed(St._queries(spans))
scores = span_vectors @ tax_vectors.T
print("score matrix:", scores.shape, "= (spans, taxonomy entries)")
by_hand = {span: St._top_matches(scores[i], entries, 3) for i, span in enumerate(spans)}

In [ ]:
show(St.standardize_constructs)
matches = standardize_constructs(relations, taxonomy="elsst", strategy="enriched", top=3)
assert [m[0]["id"] for m in matches.values()] == [by_hand[s][0]["id"] for s in matches]

In [ ]:
std_df = pd.DataFrame([{"span": span, "best concept": m[0]["leaf"], "score": m[0]["score"],
                        "2nd": m[1]["leaf"], "3rd": m[2]["leaf"]} for span, m in matches.items()])
std_df.sort_values("score", ascending=False).head(8)

In [ ]:
std_df.sort_values("score").head(8)

The first table shows spans that land near a concept. The second shows the spans with the lowest best score:
the "nearest" concept is still returned, but it is far away. Many SDT constructs have no close ELSST concept.
This is a finding of the project, not a bug, and it is the reason the package also has a taxonomy-free route
(`label_groups`, 3.8).

## 3.5 `pairwise_grouping`: group spans whose cosine reaches a threshold

The idea: draw a line between every two spans with cosine ≥ `threshold` (0.69, Paper 3's calibrated value for
all-mpnet). A **group** is then a set of spans connected by lines, directly or through other spans. In graph
terms, a group is a *connected component*.

In [ ]:
show(P.pairwise_grouping)
show(P.central_member)

By hand on the 8 toy spans from 3.1:

In [ ]:
threshold = 0.69
r_idx, c_idx = np.where(sim >= threshold)
pairs = [(toy_spans[i], toy_spans[j], round(float(sim[i, j]), 2)) for i, j in zip(r_idx, c_idx) if i < j]
print("pairs at or above", threshold, "(the diagonal and duplicates left out):")
for p in pairs:
    print("  ", p)

`np.where(sim >= threshold)` returns the row and column index of every such cell, the diagonal (a span with
itself, cosine 1) included. Those index pairs become a sparse **adjacency matrix**, and SciPy's
`connected_components` gives each span a group number:

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

adjacency = coo_matrix((np.ones(len(r_idx)), (r_idx, c_idx)), shape=(len(toy_spans), len(toy_spans)))
n_groups, labels_ = connected_components(adjacency, directed=False)
print(n_groups, "groups; label per span:", dict(zip(toy_spans, labels_)))

**The central member (medoid)** of a group is the member with the **highest mean cosine to the members**
(itself included). It is the most typical span of the group, and `label_groups` and `standardize_groups` use it.
For one group:

In [ ]:
g = labels_[toy_spans.index("well-being")]
member_idx = list(np.where(labels_ == g)[0])
sub = V[member_idx] @ V[member_idx].T
table = pd.DataFrame(sub, index=[toy_spans[i] for i in member_idx],
                     columns=[toy_spans[i] for i in member_idx]).round(2)
table["mean"] = sub.mean(axis=1).round(3)
print("central member:", toy_spans[P.central_member(member_idx, V)])
table

In [ ]:
toy_groups = pairwise_grouping(toy_spans, threshold=0.69)
assert [toy_groups[s]["group"] for s in toy_spans] == list(labels_)
toy_groups

**Chaining.** Components are transitive. If A~B and B~C, then A, B and C are one group, **even when A and C
are far apart**. On many papers this makes long chains, which is why the project notes say "read the components
before trusting them". Here is a real case from the sample spans:

In [ ]:
Vs = E.embed(spans)                       # pairwise embeds the span as it is (no lowercasing)
R = Vs @ Vs.T
chain = None
for j in range(len(spans)):
    near = [i for i in np.where(R[j] >= threshold)[0] if i != j]
    for a_ in near:
        for c_ in near:
            if a_ < c_ and R[a_, c_] < 0.5:
                chain = (a_, j, c_)
                break
        if chain:
            break
    if chain:
        break
if chain:
    a_, j, c_ = chain
    print(f"A = {spans[a_]!r}\nB = {spans[j]!r}\nC = {spans[c_]!r}")
    print(f"cos(A,B) = {R[a_, j]:.2f}   cos(B,C) = {R[j, c_]:.2f}   cos(A,C) = {R[a_, c_]:.2f}  -> one group")
else:
    print("no chain with cos(A,C) < 0.5 in these spans")

In [ ]:
groups_pw = pairwise_grouping(relations, threshold=0.69)
pw = collections.defaultdict(list)
for span, g in groups_pw.items():
    pw[g["group"]].append(span)
pd.DataFrame([{"group": k, "size": len(v), "central": groups_pw[v[0]]["central"], "members": v}
              for k, v in pw.items() if len(v) > 1]).sort_values("size", ascending=False).head(10)

Look at the **largest** group and read its members. Spans about increasing *and* decreasing competence, intrinsic
motivation and aspirations sit together, because each one is linked to the next by a chain of pairs above 0.69.
This is chaining. Note also that cosine similarity ignores **polarity**: *increase* X and *decrease* X are close
vectors, so they often land in the same group.

## 3.6 `clusterer`: group spans by density (HDBSCAN)

HDBSCAN looks for **dense regions** of points. A group needs at least `min_cluster_size` spans, and
`min_samples` sets how dense a point's neighbourhood must be. Spans in no dense region are **noise**
(`group = -1`). Unlike `pairwise_grouping`, a span may belong to no group. The defaults (2, 2, no UMAP) are Paper
3's calibrated configuration.

In [ ]:
show(Cl.clusterer)

**Why `metric="euclidean"` is right for cosine.** For vectors of length 1, `|a − b|² = 2 − 2·cos(a, b)`. The
Euclidean distance is a strictly decreasing function of the cosine, so it ranks every pair of spans in the same
order. HDBSCAN only uses that order, and its tree search is fast for the Euclidean metric.

In [ ]:
a, b = V[3], V[4]
print("|a-b|^2      =", round(float(np.sum((a - b) ** 2)), 6))
print("2 - 2 cos    =", round(float(2 - 2 * a @ b), 6))

In [ ]:
from sklearn.cluster import HDBSCAN
hand_labels = HDBSCAN(min_cluster_size=2, min_samples=2, metric="euclidean").fit_predict(Vs.astype(np.float64))
clusters = clusterer(relations)
assert [clusters[s]["group"] for s in spans] == [int(x) for x in hand_labels]
print("the same labels as HDBSCAN run by hand")

In [ ]:
cl = collections.defaultdict(list)
for span, g in clusters.items():
    cl[g["group"]].append(span)
cluster_df = pd.DataFrame([{"group": k, "size": len(v), "central": clusters[v[0]]["central"], "members": v}
                           for k, v in cl.items() if k != -1]).sort_values("size", ascending=False)
print("noise spans (group -1), first 8:", cl[-1][:8])
cluster_df.head(10)

In the output dict a noise span is **its own central member**: `{"group": -1, "central": <itself>}`. The two
functions after this one treat each noise span as a group of one, and use the span text as its key.

## 3.7 `standardize_groups`: one taxonomy concept per group

Same as 3.4, but it scores a **group**, not a span. There are three rules (`name_by`) to turn several member
vectors into one row of scores:

| rule | score of concept *t* |
|---|---|
| `mean_sim` | the mean over members of cos(member, t) |
| `centroid` | cos(mean member vector, t), after the mean vector is rescaled to length 1 |
| `medoid` | cos(central member, t) |

In [ ]:
show(St.standardize_groups)

The **largest cluster**, scored all three ways by hand:

In [ ]:
gid = int(cluster_df.iloc[0]["group"])
members = cl[gid]
central = clusters[members[0]]["central"]
Q = E.embed(St._queries(members))
print("members:", members, "\ncentral:", central, "\n")

mean_sim = (Q @ tax_vectors.T).mean(axis=0)
v = Q.mean(axis=0)
centroid = (v / np.linalg.norm(v)) @ tax_vectors.T
medoid = E.embed(St._queries([central]))[0] @ tax_vectors.T

for name, sc in [("mean_sim", mean_sim), ("centroid", centroid), ("medoid", medoid)]:
    print(f"{name:<9}", [(m["leaf"], m["score"]) for m in St._top_matches(sc, entries, 3)])

**`mean_sim` and `centroid` always rank the concepts in the same order.** The mean of the dot products equals
the dot product with the mean: `mean_i(q_i · t) = (mean_i q_i) · t = v · t`. `centroid` divides `v · t` by
`|v|`, the same number for every concept. So the two score rows differ only by a constant factor, `|v|`, which is
at most 1 and is smaller when the members point in different directions. That is why `mean_sim` gives a mixed
group a low score: it is the conservative rule.

In [ ]:
ratio = mean_sim / centroid
print("mean_sim / centroid, for every concept: min", ratio.min().round(6), " max", ratio.max().round(6))
print("|v| (length of the mean member vector):", np.linalg.norm(v).round(6))
print("same top-20 order:", np.array_equal(np.argsort(-mean_sim)[:20], np.argsort(-centroid)[:20]))

The ratio equals `|v|` for every concept. The tiny spread in the 4th decimal comes from float32 rounding. The
`medoid` rule is different: it looks at one member only, so its ranking can differ (see its 2nd concept above).

Below, the real function is called with each rule. It returns one row per **group**: the clusters plus one
group of one for each noise span.

In [ ]:
for rule, sc in [("mean_sim", mean_sim), ("centroid", centroid), ("medoid", medoid)]:
    named = standardize_groups(clusters, name_by=rule, top=3)
    assert [m["id"] for m in named[gid]["matches"]] == [m["id"] for m in St._top_matches(sc, entries, 3)]
named[gid]

## 3.8 `label_groups`: name a group from its own words, with no taxonomy

This is the taxonomy-free route. It loads no model and reads no file. It uses only the group dict. Each group gets
**keywords** ranked by *class-based TF-IDF*: all the members of one group count as one document, and

```
score(word, group) = (count of word in the group / words in the group) × log(1 + groups / groups that hold the word)
                      └────────────── frequent inside ─────────────┘   └──────────── rare outside ────────────┘
```

A word the members repeat scores high. A word that many groups use (*increased*, *motivation*) is pushed down.
Function words (`STOPWORDS`) never enter a label. The spans themselves are never changed.

In [ ]:
show(L._content_words)
show(L.label_groups)

By hand for the same largest cluster: first the rarity of each word over **all** groups (a noise span counts as
a group of one), then the score inside this group:

In [ ]:
members_all = {}
for span, g in clusters.items():
    members_all.setdefault(span if g["group"] == -1 else g["group"], []).append(span)
n_groups = len(members_all)

groups_with_word = collections.Counter()
for group_spans in members_all.values():
    groups_with_word.update(set(w for s in group_spans for w in L._content_words(s)))

for s in members:
    print(f"{s!r:<45} -> {L._content_words(s)}")

counts = collections.Counter(w for s in members for w in L._content_words(s))
n_words = sum(counts.values())
rows_kw = [{"word": w, "count in group": c, "tf = count/words": round(c / n_words, 3),
            "groups with word": groups_with_word[w],
            "rarity = log(1+G/df)": round(math.log(1 + n_groups / groups_with_word[w]), 3),
            "score": round(c / n_words * math.log(1 + n_groups / groups_with_word[w]), 3)}
           for w, c in counts.items()]
print(f"\nG = {n_groups} groups, {n_words} content words in this group")
pd.DataFrame(rows_kw).sort_values("score", ascending=False)

In [ ]:
labelled = label_groups(clusters, by="both", top=3)
top3 = sorted(rows_kw, key=lambda r: -r["score"])[:3]
assert [w for w, _ in labelled[gid]["keywords"]] == [r["word"] for r in top3]
labelled[gid]

In [ ]:
for by in L.RULES:
    print(f"{by:<9}", repr(label_groups(clusters, by=by, top=3)[gid]["name"]))

---
# Part 4 · The whole chain on one page

Everything above, as a user would write it. Each call's input is the previous call's output. The results are
dicts keyed by the original span, so everything can be traced back to `rel_id` → `sent_id` → page.

In [ ]:
# sentences = harvest("sample_files")["sentences"]        # a folder; above we used 60 sentences
relations = causenet(sentences)                            # [{"rel_id", "sent_id", "cause", "effect", ...}]

per_span  = standardize_constructs(relations, top=3)       # {span: [{"id","leaf","path","score"}, ...]}
groups_a  = clusterer(relations)                           # {span: {"group", "central"}}
groups_c  = pairwise_grouping(relations, threshold=0.69)   # {span: {"group", "central"}}
by_tax    = standardize_groups(groups_a, name_by="mean_sim", top=3)   # {group: {..., "matches"}}
by_words  = label_groups(groups_a, by="both", top=3)                  # {group: {..., "keywords", "name"}}

r = relations[0]
print("relation     :", r["rel_id"], "|", r["cause"], "->", r["effect"])
for side in ("cause", "effect"):
    span = r[side]
    g = groups_a[span]["group"]
    key = span if g == -1 else g
    print(f"{side:<7} span  : {span!r}")
    print(f"   taxonomy : {per_span[span][0]['leaf']} ({per_span[span][0]['score']})")
    print(f"   cluster  : {g} -> label {by_words[key]['name']!r}, taxonomy {by_tax[key]['matches'][0]['leaf']!r}")

## 4.1 Cheat sheet: what each function takes and returns

| function | input | output |
|---|---|---|
| `harvest(source, pages, skip_pages, steps, min_words, max_words, cache)` | PDF path or folder | `{"sentences": [{doc_id, page, sent_id, section, raw, clean, flag}], "dropped": [{..., drop_reason, text}]}` |
| `causenet(sentences, threshold, mode, decision, batch_size)` | sentence rows **or** strings | `[{rel_id, sent_id, doc_id, page, cause, effect, sentence}]` |
| `standardize_constructs(items, taxonomy, strategy, top, embeddings, leaves_only)` | relations **or** spans | `{span: [{id, leaf, path, score}] }` |
| `pairwise_grouping(items, embeddings, threshold)` | relations **or** spans | `{span: {group, central}}` |
| `clusterer(items, embeddings, min_cluster_size, min_samples, umap)` | relations **or** spans | `{span: {group, central}}`, `group = -1` is noise |
| `standardize_groups(groups, name_by, taxonomy, ...)` | group dict | `{group_id or noise span: {central, size, members, matches}}` |
| `label_groups(groups, by, top)` | group dict | `{group_id or noise span: {central, size, members, keywords, name}}` |

Not written yet: `theorize()` (relations + names → a networkx graph), `show()`, and the evaluation against
`Step 2.csv`.

## 4.2 Things this walkthrough turned up

These are behaviours of the current code that you see only when you run it step by step. None of them is fixed
here. They are listed so you know about them.

1. **The `long` flag never reaches the output.** `drop_junk_sentences` flags a sentence with more than
   `LONG_FLAG = 80` words and keeps it. `filter_length` then drops everything above `max_words = 80`. With the
   defaults, every flagged sentence is dropped as `too_long` (section 1.5, step 10). This goes against the rule
   "long sentences are flagged and kept". To keep them, raise `max_words` or remove `filter_length` from `steps`.
2. **A blank line inside a citation splits the sentence** (section 1.7). `(Ryan & Deci,⏎⏎2019)` leaves a half
   citation in one sentence, and `2019).` is dropped as `no_lowercase`. The merge rule only joins a next part that
   starts lowercase. The drop log records it.
3. **`tidy_span` repairs the straight apostrophe only** (section 2.5). This is safe after `harvest`, which
   changes `’` to `'`. It is not safe when `causenet` gets raw strings with curly quotes.
4. **"this volume" citations survive** `strip_citations`, because rule 1 of `_is_citation` needs a year (section 1.3).
5. **A running head glued to the end of a paragraph line survives** (section 1.7). The extractor sometimes puts
   the page header on the same line as the text, and `drop_running_heads` only compares whole lines. The author
   names then end up inside a kept sentence, and `causenet` sees them.
6. **About 6 % of the relations are not verbatim in their sentence** (section 2.6). The cause is spacing that
   the tokenizer lost or added (`and / or`, `(mal) adaptative`, `humans'integrative`), not missing words.
7. **`harvest` writes one file**, the page cache `<pdf>.pages.json`, although its docstring says it writes none
   (section 1.6).